In [1]:
!pip install unsloth trl transformers accelerate datasets -q

In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
import unsloth
print("Unsloth:", unsloth.__version__)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.5.2


In [3]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-e2b-it",
    max_seq_length  = 512,       # reduced — your p95 will likely be under 400
    load_in_4bit    = True,
    dtype           = None,
    device_map      = {"": 0},   # ← pin everything to GPU 0
)
print("Model loaded:", next(model.parameters()).dtype)

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Model loaded: torch.float16


In [4]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = "none",
    random_state               = 42,
    use_gradient_checkpointing = "unsloth",  # ← added, saves VRAM
)
print("LoRA adapters applied")
model.print_trainable_parameters()

LoRA adapters applied
trainable params: 25,337,856 || all params: 5,148,515,872 || trainable%: 0.4921


In [7]:
import json
from datasets import Dataset
from collections import Counter

def load_and_clean_jsonl(path):
    clean = []
    removed_hash = 0
    removed_csv  = 0

    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            user_msg  = ex["messages"][1]["content"]
            asst_data = json.loads(ex["messages"][2]["content"])

            if "condition #" in asst_data.get("alert_reason", ""):
                removed_hash += 1
                continue

            is_natural = any(x in user_msg for x in [
                "My son", "My daughter", "My child",
                "years old", "months old", "symptoms:"
            ])
            if "," in user_msg and "_" in user_msg and not is_natural:
                removed_csv += 1
                continue

            clean.append(ex)

    print(f"  Loaded   : {len(clean) + removed_hash + removed_csv}")
    print(f"  Removed 'condition #N' : {removed_hash}")
    print(f"  Removed raw CSV inputs : {removed_csv}")
    print(f"  Clean kept : {len(clean)}")
    return clean

print("=== TRAIN ===")
train_raw = load_and_clean_jsonl("/kaggle/input/datasets/jaiganeshscse/earlyeyes-training-data/earlyeyes_train.jsonl")
print("\n=== TEST ===")
test_raw  = load_and_clean_jsonl("/kaggle/input/datasets/jaiganeshscse/earlyeyes-training-data/earlyeyes_test.jsonl")

def format_gemma4(example):
    msgs = example["messages"]
    system_content = ""
    user_content   = ""
    asst_content   = ""

    for m in msgs:
        if m["role"] == "system":
            system_content = m["content"]
        elif m["role"] == "user":
            user_content = m["content"]
        elif m["role"] == "assistant":
            asst_content = m["content"]

    combined_user = f"{system_content}\n\n{user_content}" if system_content else user_content

    text = (
        f"<bos><start_of_turn>user\n"
        f"{combined_user}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{asst_content}<end_of_turn>\n"
    )
    return {"text": text}

train_formatted = [format_gemma4(e) for e in train_raw]
test_formatted  = [format_gemma4(e) for e in test_raw]

train_dataset = Dataset.from_list(train_formatted)
test_dataset  = Dataset.from_list(test_formatted)

print(f"\n✅ Train: {len(train_dataset)} examples")
print(f"✅ Test : {len(test_dataset)} examples")

alert_counts = Counter()
for ex in train_raw:
    al = json.loads(ex["messages"][2]["content"]).get("alert_level", "?")
    alert_counts[al] += 1
print(f"\nAlert level distribution:")
for k, v in alert_counts.most_common():
    print(f"  {k}: {v} ({v/len(train_raw)*100:.1f}%)")

=== TRAIN ===
  Loaded   : 24919
  Removed 'condition #N' : 6324
  Removed raw CSV inputs : 0
  Clean kept : 18595

=== TEST ===
  Loaded   : 2769
  Removed 'condition #N' : 719
  Removed raw CSV inputs : 0
  Clean kept : 2050

✅ Train: 18595 examples
✅ Test : 2050 examples

Alert level distribution:
  🟠 See a health worker soon: 9395 (50.5%)
  🔴 See a doctor today: 3564 (19.2%)
  🟡 Watch and monitor: 3179 (17.1%)
  🟢 Looking okay: 2457 (13.2%)


In [10]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir                  = "/kaggle/working/earlyeyes-gemma4",
    num_train_epochs            = 1,           # reduced from 3
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 2,
    warmup_steps                = 50,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    max_seq_length              = 256,
    dataset_text_field          = "text",
    dataset_num_proc            = 1,
    logging_steps               = 25,
    save_strategy               = "epoch",
    save_total_limit            = 1,
    eval_strategy               = "epoch",
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    seed                        = 42,
    report_to                   = "none",
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,
    packing                     = True,        # ← biggest speedup
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    args          = sft_config,
)
print("Trainer ready")
print(f"Steps per epoch : {len(trainer.get_train_dataloader())}")

Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/18595 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2050 [00:00<?, ? examples/s]

Trainer ready
Steps per epoch : 2325


In [11]:
trainer_stats = trainer.train()
print(f"\nRuntime : {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"Loss    : {trainer_stats.metrics['train_loss']:.4f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,595 | Num Epochs = 1 | Total steps = 1,163
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Epoch,Training Loss,Validation Loss
1,0.000000,3.686443


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/earlyeyes-gemma4/checkpoint-1163/tokenizer_config.json.



Runtime : 85.0 min
Loss    : 0.0000


In [13]:
from unsloth import FastModel
FastModel.for_inference(model)

test_prompt = """My son is 14 months old. \
The arms look like thin sticks — I can almost wrap my fingers all the way around. \
The ribs are clearly visible through the skin. \
The child has very little energy. Should I be worried about my child's nutrition?"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are EarlyEyes, an offline child health early warning assistant. A caregiver describes what they can see about their child — visible signs, appearance, behavior, and optionally basic measurements like weight and height. Your job is to identify warning signs of malnutrition or illness and tell the caregiver clearly what to do next. Respond ONLY in this exact JSON format with no extra text: {\"alert_level\": \"one of: See a doctor today / See a health worker soon / Watch and monitor / Looking okay\", \"alert_reason\": \"plain language explanation\", \"what_you_noticed\": [\"sign 1\"], \"what_to_do_now\": \"action\", \"tell_your_doctor\": \"what to say\", \"while_you_wait\": \"home care\", \"disclaimer\": \"This app cannot diagnose your child. Only a doctor or health worker can do that.\"}\n\n" + test_prompt
            }
        ]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    temperature    = 0.1,
    do_sample      = True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(response)

import json
try:
    parsed = json.loads(response)
    print("\n✅ Valid JSON")
    print(f"   Alert level : {parsed['alert_level']}")
    print(f"   Noticed     : {parsed['what_you_noticed']}")
except json.JSONDecodeError as e:
    print(f"\n⚠️  JSON parse error: {e}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


```json
{
  "alert_level": "See a doctor today",
  "alert_reason": "Your son's arms are very thin, and his ribs are visible. This suggests potential malnutrition.",
  "what_you_noticed": [
    "Arms look very thin",
    "Ribs are clearly visible through the skin",
    "Child has very little energy"
  ],
  "what_to_do_now": "See a doctor today.",
  "tell_your_doctor": "My son is 14 months old. His arms are very thin, and his ribs are visible through the skin. He has very little energy.",
  "while_you_wait": "Focus on providing nutritious foods that are easy for him to eat, such as soft foods and liquids. Encourage him to play gently and rest often.",
  "disclaimer": "This app cannot diagnose your child. Only a doctor or health worker can do that."
}
```

⚠️  JSON parse error: Expecting value: line 1 column 1 (char 0)


In [14]:
model.save_pretrained("/kaggle/working/earlyeyes-gemma4-lora")
tokenizer.save_pretrained("/kaggle/working/earlyeyes-gemma4-lora")
print("✅ Saved")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/earlyeyes-gemma4-lora/tokenizer_config.json.


✅ Saved


In [ ]:
# ── UPLOAD TO HUGGINGFACE ──────────────────────────────────────────────────────
from huggingface_hub import HfApi, login

# Paste your token here
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

api = HfApi()

# Create repo and upload
api.create_repo(
    repo_id="Jaiganesh1607/earlyeyes-gemma4-e2b",
    repo_type="model",
    exist_ok=True,
    private=False
)

api.upload_folder(
    folder_path="/kaggle/working/earlyeyes-gemma4-lora",
    repo_id="Jaiganesh1607/earlyeyes-gemma4-e2b",
    repo_type="model",
)

print("✅ Model uploaded to HuggingFace")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model uploaded to HuggingFace
